# 📱 DjezzyBot — Colab Runbook (T4 GPU)

A voice + text chatbot for Djezzy, in Arabic / French / English / Darija.

**Do this first:** `Exécution ▸ Modifier le type d'exécution ▸ T4 GPU`.

Then run the steps below from top to bottom. The code is pulled straight from
GitHub (`YacefMehdi/DjezzyBot`); Step 1 clones it the first time and pulls the
latest commit on every run after — no Drive upload needed.

> At startup the bot answers from the cached `data/djezzy_pages.json`. The live
> crawler (the **Rafraîchir** button and the daily 03:00 job) re-scrapes djezzy.dz
> with Playwright/Chromium — Step 1 installs that browser; Step 1b verifies it.


## Step 1 - Setup  (pull latest code from a private GitHub repo + install)

> **One-time:** the repo is private, so add a read-only GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar, enable *Notebook access*). The cell below explains exactly how.

> **To apply an update:** `git push` from your machine, then just **re-run this cell** - it pulls the newest commit. If the session was already running when you pushed, do **Execution > Restart session** first, then run top to bottom, so Python loads the *new* code and not the old copy still in memory.


In [ ]:
# Setup: pull the latest code from the PRIVATE GitHub repo, then install deps.
# No more Drive upload/delete. The repo is the single source of truth: this cell
# CLONES it the first time and PULLS the newest commit every run after. To update
# the bot, just `git push` from your machine, then re-run this cell in Colab.
#
# The repo is PRIVATE, so Colab needs a read-only GitHub token. Store it ONCE:
#   left sidebar > key icon (Secrets) > "Add new secret"
#       Name:  GH_TOKEN
#       Value: a fine-grained Personal Access Token (Contents: Read-only) for this repo
#   then turn "Notebook access" ON for that secret.
#   Make a token at: GitHub > Settings > Developer settings > Personal access tokens
#                    > Fine-grained tokens > Generate (Only select repositories: DjezzyBot,
#                      Repository permissions > Contents: Read-only).
import os, sys, shutil, subprocess
from google.colab import userdata

OWNER, REPO_NAME = "YacefMehdi", "DjezzyBot"
PROJ = "/content/DjezzyBot"
CLEAN_URL = f"https://github.com/{OWNER}/{REPO_NAME}.git"

try:
    token = userdata.get("GH_TOKEN")
except Exception as e:
    raise SystemExit(
        "No GH_TOKEN secret found. Open the key icon in the left sidebar, add a secret "
        "named GH_TOKEN holding a read-only GitHub token, enable Notebook access, and re-run. "
        f"({type(e).__name__})")
auth_url = f"https://{token}@github.com/{OWNER}/{REPO_NAME}.git"

def _run(cmd):
    # capture output and, on failure, surface the REAL error with the token redacted
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError((r.stderr or r.stdout).replace(token, "***"))
    return r.stdout

if os.path.isdir(os.path.join(PROJ, ".git")):
    print("Pulling the latest commit...")
    _run(f"git -C {PROJ} reset --hard -q")          # discard in-Colab edits; GitHub wins
    _run(f"git -C {PROJ} pull -q {auth_url} main")
else:
    if os.path.isdir(PROJ):
        shutil.rmtree(PROJ)                          # clean up a partial/failed earlier clone
    print("Cloning the repo (first run)...")
    _run(f"git clone -q {auth_url} {PROJ}")
    _run(f"git -C {PROJ} remote set-url origin {CLEAN_URL}")  # don't persist the token on disk

os.chdir(PROJ); sys.path.insert(0, PROJ)
print("Code up to date at", os.getcwd(), "-", _run(f"git -C {PROJ} log -1 --oneline").strip())

subprocess.run("pip install -q -r requirements.txt", shell=True)

# Playwright's CHROMIUM BINARY is NOT installed by pip - without it the live scraper
# and the daily refresh (scraper.run_scrape / the Refresh button) fail. Install it now.
print("Installing Chromium for the scraper... (one-time, ~150 MB)")
subprocess.run("playwright install chromium", shell=True)
subprocess.run("playwright install-deps chromium", shell=True)  # system libs (Colab is root)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - set Execution > Change runtime type > T4 GPU, then re-run")


## Step 1b — Verify the scraper  (optional)
Confirms Chromium is installed and the crawler can render a live djezzy.dz page, so the **Rafraîchir** button and the daily 03:00 refresh will actually work.

In [ ]:
# Step 1b — verify the scraper stack (optional, ~10 s). Renders ONE live page to
# confirm Chromium works, so you know the Refresh button / daily job will run.
import scraper
print(scraper.smoke_test())   # -> {'ok': True, 'url': ..., 'title': ..., 'chars': ...}

# Full refresh (re-crawls the whole site, up to ~45 min, then rebuilds the index):
#   import scheduler
#   print(scheduler.force_refresh())     # -> {'ok': True, 'n_pages': N, ...}
#   store = scheduler.CURRENT_INDEX      # hot-swapped fresh index

## Step 2 — Build the search index  (from the saved data)

In [ ]:
import scraper, indexer
pages = scraper.load_pages()
assert pages, "data/djezzy_pages.json is missing — make sure it came with the project."
store = indexer.build_index(pages)
print(f"✅ Indexed {store.index.ntotal} chunks from {len(pages)} pages.")

## Step 3 — Test it  (optional, takes a few minutes)
Runs the 14 acceptance checks and prints a PASS/FAIL summary.

In [ ]:
import bot, test_scenarios
results = test_scenarios.run_all(store)
test_scenarios._summary(results)

## Step 3b — Robustness (stress) suite  (optional, ~10 min)
25 harder cross-lingual scenarios (budget, out-of-domain, comparison, roaming, named offers…) across Arabic / French / English / Darija. Surfaces weak spots; this is **not** the acceptance contract.

In [ ]:
# Robustness / stress suite: 25 harder, mostly cross-lingual scenarios that probe
# every route in Arabic / French / English / Darija. Failures here are engineering
# findings to investigate, NOT a broken acceptance contract.
import test_robustness
test_robustness._summary(test_robustness.run_all(store))

## Step 3c — Quantitative metrics  (the real numbers for the thesis)
Per-class precision / recall / F1 for language detection and intent routing, retrieval **recall@k vs a no-router baseline** (the router ablation), the **OOD domain gate**, and automatic answer **groundedness**. The LLM sections print live progress.

In [ ]:
# Quantitative metrics: per-class precision/recall/F1 for language & routing,
# retrieval recall@k vs a no-router baseline, and — with the LLM — the OOD domain
# gate + answer groundedness. Sections 4-5 print live per-generation progress
# (~22 generations, ~10-12 min on a T4); the per-line output means it is NOT a hang.
import evaluate
metrics = evaluate.run_all(store, with_llm=True)

## Step 4 — Launch the app
Opens the chatbot (text + voice). Click the public **share** link it prints.

In [ ]:
import app
app.main()